# P2 · Survival Filter
**Input:** `data/processed/dea_results.csv` · `results/tables/hub_genes.csv`  
**Outputs:** `results/tables/survival_filtered_genes.csv` · `results/tables/survival_results.csv`
· `results/figures/km_plots.png` · `results/figures/cox_forest_plot.png`

Downloads TCGA-LIHC data (n≈374 patients). Falls back to simulated data if unavailable.

**Run order:** P1 → **P2** → P3 → P4


In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    for p in [start, *start.parents]:
        if (p / "paths.py").exists():
            return p
    raise FileNotFoundError("paths.py not found — run: python scripts/data_download.py")

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from paths import REPO_ROOT, PROC_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR
sys.path.insert(0, str(REPO_ROOT / "scripts"))
print(f"Repo root : {REPO_ROOT}")

In [ ]:
from survival_functions import (load_gene_list, fetch_tcga_lihc,
                                simulate_tcga, run_survival,
                                filter_survivors, export_survival)
from utils import plot_km_grid, plot_cox_forest
import matplotlib.pyplot as plt
print("Imports OK")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
KM_P_THRESH   = 0.05
COX_P_THRESH  = 0.05
HR_MIN        = 0.8    # ─┐ exclude HR between these values
HR_MAX        = 1.2    # ─┘ (no meaningful prognostic effect)
TOP_KM        = 12     # genes shown in KM grid
RANDOM_SEED   = 42

## 1 · Load gene list

In [ ]:
sig, gene_list, hub_score_map = load_gene_list(
    PROC_DIR / "dea_results.csv",
    hub_path=TABLES_DIR / "hub_genes.csv",
)

## 2 · Fetch TCGA-LIHC data

In [ ]:
merged, is_sim = fetch_tcga_lihc()
if is_sim:
    merged = simulate_tcga(gene_list, random_seed=RANDOM_SEED)

avail = [g for g in gene_list if g in merged.columns]
print(f"Genes with expression data: {len(avail)}/{len(gene_list)}")
print(f"Data source: {'⚠ SIMULATED' if is_sim else '✓ TCGA-LIHC'}")

## 3 · Run Kaplan–Meier + Cox per gene

In [ ]:
surv_raw = run_survival(gene_list, merged)
surv_df, surv_filtered = filter_survivors(
    surv_raw, sig,
    km_p=KM_P_THRESH, cox_p=COX_P_THRESH,
    hr_min=HR_MIN, hr_max=HR_MAX,
)
surv_filtered[["gene","HR","prognosis","logrank_p"]].head(10)

## 4 · Kaplan–Meier grid

In [ ]:
top_genes = surv_filtered.head(TOP_KM).gene.tolist()
if len(top_genes) < TOP_KM:
    top_genes += (surv_df[~surv_df.gene.isin(top_genes)]
                  .head(TOP_KM - len(top_genes)).gene.tolist())

fig = plot_km_grid(top_genes, surv_df, merged, is_simulated=is_sim)
fig.savefig(FIGURES_DIR / "km_plots.png", dpi=200, bbox_inches="tight")
plt.show()

## 5 · Cox forest plot

In [ ]:
fig, _ = plot_cox_forest(surv_df, top_n=20, cox_p_thresh=COX_P_THRESH)
fig.savefig(FIGURES_DIR / "cox_forest_plot.png", dpi=200, bbox_inches="tight")
plt.show()

## 6 · Export

In [ ]:
export_survival(surv_df, surv_filtered, TABLES_DIR)